In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore')

print("=" * 70)
print("ETL DATA WAREHOUSE - RENT4YOU")
print("Preprocesamiento de Datos para Análisis de Ventas por Sucursal")
print("=" * 70)

print("\n" + "=" * 70)
print("FASE 1: GENERACIÓN DE DATOS CRUDOS")
print("=" * 70)

print("\n[1/5] Generando datos de Sucursales...")
sucursales_raw = pd.DataFrame({
    'id_sucursal': [1, 2, 3, 4, 5, 5, 6],
    'nombre_sucursal': ['Centro CDMX', 'Norte MTY', 'Sur GDL', 'Este Puebla', None, 'Occidente QRO', 'Bajío León'],
    'ciudad': ['CDMX', 'monterrey', 'Guadalajara', 'Puebla', 'Querétaro', 'Querétaro', '  León  '],
    'region': ['Centro', 'Norte', 'Occidente', 'Centro', 'Centro', 'Centro', 'Bajío']
})
print(f"   ✓ Generados {len(sucursales_raw)} registros (con duplicados y nulos)")

print("\n[2/5] Generando datos de Clientes...")
np.random.seed(42)
nombres = ['Ana García', 'Carlos López', 'María Rodríguez', 'José Martínez', 'Laura Hernández',
           'Miguel Pérez', 'Carmen González', 'Francisco Sánchez', 'Isabel Ramírez', 'Antonio Torres']
clientes_raw = pd.DataFrame({
    'id_cliente': list(range(1, 102)) + [50],
    'nombre': np.random.choice(nombres, 102),
    'edad': np.random.randint(18, 75, 102),
    'ciudad': np.random.choice(['CDMX', 'Monterrey', 'Guadalajara', 'Puebla', 'Querétaro', None], 102),
    'genero': np.random.choice(['M', 'F', 'Otro', None], 102),
    'tipo_cliente': np.random.choice(['Regular', 'VIP', 'Corporativo', 'Regular'], 102)
})
clientes_raw.loc[0, 'ciudad'] = None
clientes_raw.loc[25, 'genero'] = None
clientes_raw.loc[75, 'ciudad'] = None
print(f"   ✓ Generados {len(clientes_raw)} registros (con duplicados y nulos)")

print("\n[3/5] Generando catálogo de Vehículos...")
vehiculos_raw = pd.DataFrame({
    'id_vehiculo': range(1, 21),
    'tipo_vehiculo': np.random.choice(['Auto', 'Moto', 'Camioneta', 'Remolque', 'SUV'], 20),
    'marca': np.random.choice(['Toyota', 'Honda', 'Ford', 'Chevrolet', 'Nissan', 'Mazda'], 20),
    'modelo': [f'Modelo {chr(65+i)}' for i in range(20)],
    'ano_fabricacion': np.random.randint(2015, 2025, 20),
    'costo_diario': np.random.uniform(300, 2500, 20).round(2)
})
vehiculos_raw.loc[0, 'costo_diario'] = -500.00
vehiculos_raw.loc[5, 'costo_diario'] = 0
vehiculos_raw.loc[10, 'ano_fabricacion'] = 2030
print(f"   ✓ Generados {len(vehiculos_raw)} registros (con valores inválidos)")

print("\n[4/5] Generando Dimensión Tiempo...")
fecha_inicio = datetime(2023, 1, 1)
fecha_fin = datetime(2024, 12, 31)
fechas = pd.date_range(fecha_inicio, fecha_fin, freq='D')
tiempo_raw = pd.DataFrame({
    'id_fecha': range(1, len(fechas) + 1),
    'fecha': fechas,
    'mes': fechas.month,
    'trimestre': fechas.quarter,
    'ano': fechas.year,
    'dia_semana': fechas.day_name()
})
print(f"   ✓ Generados {len(tiempo_raw)} registros (calendario completo 2023-2024)")

print("\n[5/5] Generando datos de Ventas...")
np.random.seed(123)
n_ventas = 500
ventas_raw = pd.DataFrame({
    'id_venta': range(1, n_ventas + 1),
    'id_cliente': np.random.randint(1, 102, n_ventas),
    'id_vehiculo': np.random.randint(1, 21, n_ventas),
    'id_sucursal': np.random.randint(1, 7, n_ventas),
    'id_fecha': np.random.randint(1, len(fechas) + 1, n_ventas),
    'cantidad_dias': np.random.randint(1, 31, n_ventas),
    'monto_total': np.random.uniform(500, 20000, n_ventas).round(2),
    'descuento': np.random.uniform(0, 1000, n_ventas).round(2),
    'metodo_pago': np.random.choice(['Efectivo', 'Tarjeta', 'Transferencia'], n_ventas)
})
ventas_raw = pd.concat([ventas_raw, ventas_raw.iloc[[0, 1, 2]]], ignore_index=True)
ventas_raw.loc[10, 'cantidad_dias'] = -5
ventas_raw.loc[20, 'monto_total'] = -1000
ventas_raw.loc[30, 'id_cliente'] = 999
ventas_raw.loc[40, 'id_vehiculo'] = 999
ventas_raw.loc[50, 'id_sucursal'] = 999
print(f"   ✓ Generados {len(ventas_raw)} registros (con duplicados y referencias inválidas)")
print("\n✓ Datos crudos generados exitosamente")
print(f"\nTotal de registros crudos: {len(sucursales_raw) + len(clientes_raw) + len(vehiculos_raw) + len(tiempo_raw) + len(ventas_raw)}")

print("\n" + "=" * 70)
print("FASE 2: PROCESO ETL - LIMPIEZA Y TRANSFORMACIÓN")
print("=" * 70)

print("\n[1/5] Limpiando DIM_SUCURSAL...")
dim_sucursal = sucursales_raw.copy()
registros_antes = len(dim_sucursal)
dim_sucursal = dim_sucursal.drop_duplicates(subset=['id_sucursal'], keep='first')
dim_sucursal['nombre_sucursal'] = dim_sucursal['nombre_sucursal'].fillna('No especificado')
dim_sucursal['ciudad'] = dim_sucursal['ciudad'].str.strip().str.title()
dim_sucursal['region'] = dim_sucursal['region'].str.strip().str.title()
dim_sucursal['nombre_sucursal'] = dim_sucursal['nombre_sucursal'].str.strip()
print(f"   → Registros finales: {len(dim_sucursal)}")

print("\n[2/5] Limpiando DIM_CLIENTE...")
dim_cliente = clientes_raw.copy()
dim_cliente = dim_cliente.drop_duplicates(subset=['id_cliente'], keep='first')
dim_cliente['ciudad'] = dim_cliente['ciudad'].fillna('No especificado')
dim_cliente['genero'] = dim_cliente['genero'].fillna('No especificado')
dim_cliente = dim_cliente[(dim_cliente['edad'] >= 18) & (dim_cliente['edad'] <= 100)]
dim_cliente['ciudad'] = dim_cliente['ciudad'].str.strip().str.title()
dim_cliente['nombre'] = dim_cliente['nombre'].str.strip().str.title()
print(f"   → Registros finales: {len(dim_cliente)}")

print("\n[3/5] Limpiando DIM_VEHICULO...")
dim_vehiculo = vehiculos_raw.copy()
dim_vehiculo = dim_vehiculo[dim_vehiculo['costo_diario'] > 0]
ano_actual = datetime.now().year
dim_vehiculo = dim_vehiculo[(dim_vehiculo['ano_fabricacion'] >= 2000) & (dim_vehiculo['ano_fabricacion'] <= ano_actual)]
dim_vehiculo['tipo_vehiculo'] = dim_vehiculo['tipo_vehiculo'].str.strip().str.title()
dim_vehiculo['marca'] = dim_vehiculo['marca'].str.strip().str.title()
dim_vehiculo['modelo'] = dim_vehiculo['modelo'].str.strip()
print(f"   → Registros finales: {len(dim_vehiculo)}")

print("\n[4/5] Validando DIM_TIEMPO...")
dim_tiempo = tiempo_raw.copy()
dim_tiempo['fecha'] = pd.to_datetime(dim_tiempo['fecha'])
print(f"   → Registros finales: {len(dim_tiempo)}")

print("\n[5/5] Limpiando FACT_VENTAS...")
fact_ventas = ventas_raw.copy()
fact_ventas = fact_ventas.drop_duplicates()
fact_ventas = fact_ventas[fact_ventas['cantidad_dias'] > 0]
fact_ventas = fact_ventas[fact_ventas['monto_total'] > 0]
fact_ventas = fact_ventas[fact_ventas['id_cliente'].isin(dim_cliente['id_cliente'])]
fact_ventas = fact_ventas[fact_ventas['id_vehiculo'].isin(dim_vehiculo['id_vehiculo'])]
fact_ventas = fact_ventas[fact_ventas['id_sucursal'].isin(dim_sucursal['id_sucursal'])]
fact_ventas = fact_ventas[fact_ventas['id_fecha'].isin(dim_tiempo['id_fecha'])]
print(f"   → Registros finales: {len(fact_ventas)}")
print("\n✓ Proceso ETL completado exitosamente")

print("\n" + "=" * 70)
print("FASE 3: RESUMEN DE DATOS PREPROCESADOS")
print("=" * 70)
print("\nTABLAS DEL DATA WAREHOUSE:")
print(f"   1. DIM_SUCURSAL:    {len(dim_sucursal):>5} registros")
print(f"   2. DIM_CLIENTE:     {len(dim_cliente):>5} registros")
print(f"   3. DIM_VEHICULO:    {len(dim_vehiculo):>5} registros")
print(f"   4. DIM_TIEMPO:      {len(dim_tiempo):>5} registros")
print(f"   5. FACT_VENTAS:     {len(fact_ventas):>5} registros")
print(f"   {'─' * 40}")
print(f"   TOTAL:              {len(dim_sucursal) + len(dim_cliente) + len(dim_vehiculo) + len(dim_tiempo) + len(fact_ventas):>5} registros")

print("\n" + "=" * 70)
print("FASE 4: ANÁLISIS EXPLORATORIO")
print("=" * 70)
print("\nVENTAS POR SUCURSAL:")
ventas_por_sucursal = fact_ventas.merge(dim_sucursal, on='id_sucursal')
resumen_sucursal = ventas_por_sucursal.groupby(['nombre_sucursal', 'ciudad']).agg({
    'monto_total': ['sum', 'mean', 'count']
}).round(2)
resumen_sucursal.columns = ['Monto Total', 'Ticket Promedio', 'Num. Rentas']
print(resumen_sucursal)

print("\nVEHÍCULOS MÁS RENTADOS:")
vehiculos_rentados = fact_ventas.merge(dim_vehiculo, on='id_vehiculo')
top_vehiculos = vehiculos_rentados.groupby('tipo_vehiculo').agg({
    'id_venta': 'count',
    'monto_total': 'sum'
}).round(2)
top_vehiculos.columns = ['Cantidad Rentas', 'Ingresos Totales']
top_vehiculos = top_vehiculos.sort_values('Cantidad Rentas', ascending=False)
print(top_vehiculos)

print("\nPERFIL DE CLIENTES POR TIPO:")
perfil_clientes = dim_cliente.groupby('tipo_cliente').agg({
    'id_cliente': 'count',
    'edad': 'mean'
}).round(2)
perfil_clientes.columns = ['Cantidad', 'Edad Promedio']
print(perfil_clientes)

print("\n" + "=" * 70)
print("FASE 5: EXPORTACIÓN DE DATOS")
print("=" * 70)
carpeta_destino = 'datos_preprocesados'
if not os.path.exists(carpeta_destino):
    os.makedirs(carpeta_destino)
    print(f"\n✓ Carpeta '{carpeta_destino}' creada")

print("\nExportando tablas a CSV...")
dim_sucursal.to_csv(f'{carpeta_destino}/dim_sucursal.csv', index=False, encoding='utf-8-sig')
dim_cliente.to_csv(f'{carpeta_destino}/dim_cliente.csv', index=False, encoding='utf-8-sig')
dim_vehiculo.to_csv(f'{carpeta_destino}/dim_vehiculo.csv', index=False, encoding='utf-8-sig')
dim_tiempo.to_csv(f'{carpeta_destino}/dim_tiempo.csv', index=False, encoding='utf-8-sig')
fact_ventas.to_csv(f'{carpeta_destino}/fact_ventas.csv', index=False, encoding='utf-8-sig')
print(f"\n✓ Todos los archivos exportados exitosamente a '{carpeta_destino}/'")

print("\n" + "=" * 70)
print("FASE 6: VALIDACIÓN FINAL DE CALIDAD")
print("=" * 70)
print("\nCHECKLIST DE CALIDAD:")
print(f"   ✓ Duplicados eliminados en todas las tablas")
print(f"   ✓ Valores nulos manejados correctamente")
print(f"   ✓ Rangos validados (edades, costos, fechas)")
print(f"   ✓ Integridad referencial verificada")
print(f"   ✓ Normalización de texto aplicada")
print(f"   ✓ Consistencia de datos garantizada")
print("\n" + "=" * 70)
print("✓ PROCESO ETL COMPLETADO EXITOSAMENTE")
print("=" * 70)
print("\nArchivos generados:")
print(f"   → {carpeta_destino}/dim_sucursal.csv")
print(f"   → {carpeta_destino}/dim_cliente.csv")
print(f"   → {carpeta_destino}/dim_vehiculo.csv")
print(f"   → {carpeta_destino}/dim_tiempo.csv")
print(f"   → {carpeta_destino}/fact_ventas.csv")
print("\nData Warehouse listo para análisis de Business Intelligence")
print("=" * 70)


ETL DATA WAREHOUSE - RENT4YOU
Preprocesamiento de Datos para Análisis de Ventas por Sucursal

FASE 1: GENERACIÓN DE DATOS CRUDOS

[1/5] Generando datos de Sucursales...
   ✓ Generados 7 registros (con duplicados y nulos)

[2/5] Generando datos de Clientes...
   ✓ Generados 102 registros (con duplicados y nulos)

[3/5] Generando catálogo de Vehículos...
   ✓ Generados 20 registros (con valores inválidos)

[4/5] Generando Dimensión Tiempo...
   ✓ Generados 731 registros (calendario completo 2023-2024)

[5/5] Generando datos de Ventas...
   ✓ Generados 503 registros (con duplicados y referencias inválidas)

✓ Datos crudos generados exitosamente

Total de registros crudos: 1363

FASE 2: PROCESO ETL - LIMPIEZA Y TRANSFORMACIÓN

[1/5] Limpiando DIM_SUCURSAL...
   ✓ Duplicados eliminados: 1
   ✓ Valores nulos manejados: 1
   ✓ Normalización de texto completada
   → Registros finales: 6

[2/5] Limpiando DIM_CLIENTE...
   ✓ Duplicados eliminados: 1
   ✓ Valores nulos manejados: 48
   ✓ Validaci